In [ ]:
import pathlib
import re

In [ ]:
import mywidgets.Tools.uiStyle.MarkupRe as MarkupRe
import excelxml
from excelxml import OXLLoader, regex_range, unescape, excel_col_to_int, wscell_address, offset_rng, wscell_pattern

In [ ]:
from collections.abc import Callable
from typing import Any

In [ ]:
def exec_time(
    f1: Callable[..., Any],
    f2: Callable[..., Any],
    *args: Any,
    **kwargs: Any,
) -> None:
    '''
    Helper function that benchmarks two functions with the same arguments
    '''

    from statistics import median
    from timeit import repeat

    number = 1_000
    ntimes = 7

    # Check they return equivalent results before timing.
    assert f1(*args, **kwargs) == f2(*args, **kwargs)

    for fnc in (f1, f2):
        f_times = repeat(lambda f=fnc: f(*args, **kwargs), number=number, repeat=ntimes)
        print(f"{fnc.__name__}:    {median(f_times) / number * 1_000_000:.2f} µs/call")


In [ ]:
fpath = pathlib.Path(r'C:\Users\agmontesb\Documents\DIAN\Renta2025\F2517_AG2025')
fname = 'Reporte_Conciliación_Fiscal_F2517V8_AG2025_v1.0.0-2026.xlsm'
full_path = fpath / fname


In [ ]:
loader = OXLLoader(full_path)

In [ ]:
fout = r'C:\Users\agmontesb\Documents\GitHub\excel\tests\files\h3_eri_renta_liquida.xml'
# with open(fout, 'w') as f:
#     f.write(h3_xml)

In [ ]:
from excel_workbook import ExcelWorkbook, ExcelWorksheet, ExcelTable, pythonize_fml

In [ ]:
wb = excelxml.WorkBookXml(full_path)
ws = wb.active
ws.title
# wb.load_wb_data(full_path)


In [ ]:
nrow = 378
row = ws[f'{nrow}:{nrow}']
row

In [ ]:
ws.cell(378, 1, 378, 14)

In [ ]:
list(ws.iter_rows(378, 1, 378, 14))

In [ ]:
exec_time(ws.cell_new, ws.cell, 378, 1, 378, 14)

In [ ]:
fnc = lambda x: '{0: >4s}{1: >4s}'.format(*excelxml.wscell_address(x))

In [ ]:
gnc = lambda x: any(map(x.startswith, tuple('GHIJKLMN')))
gnc('N23')

In [ ]:
ws_range = 'H9:N671'
cells = ws[ws_range]

In [ ]:
values = {key: cell.value for key, cell in cells.items() if cell.value is not None}
fmls = {key: cell.formula for key, cell in cells.items() if cell.formula}

In [ ]:
row = {key: value for key, value in cells.items() if gnc(key) and key.endswith('378')}
{key: row[key] for key in sorted(row.keys(), key=lambda x: fnc(x))}

In [ ]:
ws_name = ws.title
ws_name, full_path, ws_range

In [ ]:
excel_wb = ExcelWorkbook(full_path)
excel_wb.create_worksheet('H4 (Impuesto Diferido)')
wsheet = excel_wb.create_worksheet(ws_name)


In [ ]:
cells = ws[ws_range]
fmls1 = {adr: f"={fml.lstrip('=+')}" for adr, cell in cells.items() if (fml := cell.formula_)}
vals1 = {adr: cell.value for adr, cell in cells.items() if adr not in fmls1}

In [ ]:
# cells = ws_data['cells']
keys = sorted(cells, key=lambda x:fnc(x))
fmls = {key: f"={cells[key][1].lstrip('=+')}" for key in keys if gnc(key) and cells[key][1] is not None}
all_values = {key: cells[key][2] for key in keys if gnc(key) and not isinstance(cells[key][2], str)}


In [ ]:
values = {key: all_values[key] for key in (all_values.keys() - fmls.keys())}


In [ ]:
diff = vals1.keys() - values.keys()
{key: cells[key] for key in diff}


In [ ]:
fmls['K595']

In [ ]:
pythonize_fml(fmls['K595'], table_name='tbl')

In [ ]:
sht_tbl = ExcelTable(wsheet, 'Renta_liquida', ws_range, fmls=fmls, values=values)


In [ ]:
sht_tbl.parent.parameters(M6=49799)

In [ ]:
mtable = sht_tbl.minimun_table(any_row=False)
mtable

In [ ]:
wname = ws.title
ws_path = wb._sheet_names[wname]
h3_xml = wb.zf.load_xmlfile(ws_path, isAbs=False)
h3_xml[:100]

In [ ]:
h3_xml[352862:352898]

In [ ]:
all_values['L418']
[x for x in fmls.keys() if x[1:] == '418']
ws['L418']

In [ ]:
mask = mtable.any(axis=1)
mtable[mask]


In [ ]:
tbl = sht_tbl

In [ ]:
tbl.get_formula('383')

In [ ]:
tbl.set_records({'J383': 683400}, 'value')

In [ ]:
tbl.recalculate(True)

In [ ]:
dmy = tbl.minimun_table()
mask = dmy.any(axis=1)
dmy[mask]

In [ ]:
pythonize_fml(fmls['L386'], table_name='tbl')

In [ ]:
mtable.loc['386']

In [ ]:
sht_tbl.set_records(values, field='value')


In [ ]:
sht_tbl.set_records(fmls, field='fml')

In [ ]:

# Acá se hace una corrección ya que la celda 'G142' es un valor pero se encuentra expresada como formula 
# =151647+1
fmls.pop('G142')
all_values['G142'] = 151_647 + 1


In [ ]:
ws_name = wb.sheetnames[12]
wsheet = excel_wb.create_worksheet(ws_name)
sht_tbl = ExcelTable(wsheet, 'ESF_Patrimonio', 'G11:K227', fmls=fmls, values=values)


### BORRADOR

In [1]:
import pandas as pd

In [2]:
from xlpatterns import rgn_pattern, cell_pattern, code_alpha, tbl_pattern, tbl_address
from xlobjects import XlRef, EMPTY_CELL, XlErrors
from tests.fixtures import static_workbook
from tests.utilities import unwrapped_fixture, exec_time, TableComparator
from excel_workbook import ExcelTable, ExcelWorksheet, ExcelWorkbook, pythonize_fml

In [3]:
import itertools

In [4]:
# bw = base_workbook = unwrapped_fixture(static_workbook)()

In [5]:
df = (
    pd.Series({'B15': "'sheet1'!A1", "B40": "'sheet1'!D20"})
    .to_frame('code')
    .rename_axis("cell")
)

In [6]:
fml = "+SUM('sheet1'!A1:D20)"

In [7]:
ExcelTable.encoder('decode', fml, df)

'+SUM(B15:B40)'

In [8]:
wb = ExcelWorkbook('MyBook')
sht1 = wb.create_worksheet('Sht1')
tbl1 = sht1.create_table('tbl1', 'A1:P100')
sht2 = wb.create_worksheet('Sht2')
tbl2 = sht2.create_table('tbl2', 'A1:P100')


In [9]:
tbl1.set_records({'A1': "=SUM('Sht2'!A1:A10)"})

In [10]:
tbl1.recalculate()

In [11]:
tbl2.data

,fml,dependents,res_order,ftype,value,code
cell,,,,,,
A1,,{},0,?,,A1
A10,,{},0,?,,A2


In [12]:
tbl2.set_values({'A10':20}, recalc=True)

In [14]:
# tbl2.cells(10, 'A')
assert tbl2.cells(10, 'A').value == 20
assert tbl1.cells(1, 'A').formula == "=SUM('Sht2'!A1:A10)"
assert tbl1.cells(1, 'A').value == 20


In [12]:
tbl1.set_formulas({'A2': '=mio(23)'})

ValueError: not enough values to unpack (expected 2, got 0)

In [13]:
tbl1.data

,fml,dependents,res_order,ftype,value,code
cell,,,,,,
A1,='sheet2'!A2 + 'sheet2'!A1,{},1,$,0,A1
'Sht2'!A1,,{A1},0,#,,'sheet2'!A2
'Sht2'!A10,,{A1},0,#,,'sheet2'!A1


In [20]:
assert sht1.parameters() == []
assert sht2.parameters() == ['A1', 'A10'], 'Los enlaces a celdas no existentes en otras tablas se crean como paramaters'

In [5]:
        wb = ExcelWorkbook('MyBook')
        sht1 = wb.create_worksheet('Sht1')
        tbl1 = sht1.create_table('tbl1', 'A1:P100')
        sht2 = wb.create_worksheet('Sht2')
        tbl2 = sht2.create_table('tbl2', 'A1:P100')

        tbl1.set_formulas({'A1': "='Sht2'!A1 + 'Sht2'!A10"}, recalc=True)
        tbl2.set_values({'A10':20}, recalc=True)


In [ ]:

        assert tbl2.data.loc['A10'].value == 20
        assert tbl1.data.loc['A1'].fml == "='sheet2'!A2 + 'sheet2'!A1"
        assert tbl1.data.loc['A1'].value == 20


In [10]:
        df = tbl1.data
        eqv_tbl = dict(zip(df.code, df.index))


In [11]:
        new_df = ExcelTable.set_field(eqv_tbl, field='code', data=df, with_dependents=False)


In [34]:
        eqv_tbl = df.code.to_dict()
        new_df = ExcelTable.set_field(eqv_tbl, field='code', data=df, with_dependents=False)


In [38]:
assert (new_df.code == df.code).all()

In [14]:
it = cell_pattern.finditer("='sheet2'!A2 + 'sheet2'!A1")
for m in it:
    print(m[0], m.groups())

'sheet2'!A2 ('sheet2', 'A', '2')
'sheet2'!A1 ('sheet2', 'A', '1')


In [9]:
tbl1.minimum_table(False)

col,A,B,C,D,E,F,G,H,I,J,K,L,M,N,O,P
row,,,,,,,,,,,,,,,,


In [19]:
fmls = {'A1': "=SUM('Sht2'!A1:A10)"}
fmls_df = pd.Series(fmls).to_frame(name='fml')
fmls_df.index.name = 'cell'
fmls_df['code'] = ['A1']
fmls_df

,fml,code
cell,,
A1,=SUM('Sht2'!A1:A10),A1


In [22]:
code_ndx = {'A1': 'A1', "'sheet2'!A1": "'Sht2'!A1", "'sheet2'!A10": "'Sht2'!A10"}
code_ndx = {value: key for key, value in code_ndx.items()}

In [23]:
ExcelTable.set_field(code_ndx, field='code', data=fmls_df, with_dependents=False)

,fml,code
cell,,
A1,=SUM('sheet2'!A1:A10),A1


In [3]:
basic_wb = unwrapped_fixture(static_workbook)()

In [4]:
wb = ExcelWorkbook('MyBook')
sht1 = wb.create_worksheet('Sht1')
tbl1 = sht1.create_table('tbl1', 'A1:P100')
sht2 = wb.create_worksheet('Sht2')
tbl2 = sht2.create_table('tbl2', 'A1:P100')

tbl2.set_values({'A1':0, 'A10':0})


In [5]:
tbl1.data

,fml,dependents,res_order,ftype,value,code
cell,,,,,,


In [7]:
tbl1.set_formulas({'A1': "=SUM('Sht2'!A1:A10)"})


KeyError: 'A10'

In [3]:
def xlslice_in_fml(cell, fml):
    if not fml:
        return False
    components = set(rgn_pattern.findall(fml))
    if cell in components:
        return True
    xlslice_components = [XlRef(x) for x in components if ':' in x]
    return any(cell in xlslice for xlslice in xlslice_components)

In [4]:
def is_cell_in_fml(cell, fml):
        code = lambda cell: '{1:0>6s}|{0:_>3s}'.format(*cell_pattern.match(cell).groups()[1:]).split('|')
        if not fml:
            return False
        components = [x.replace('$', '').split(':') for x in set(rgn_pattern.findall(fml))]
        if not any(x == cell for x in itertools.chain(*components)):
            cell_code = code(cell)
            rng_limits = (
                [code(x) for x in pair]
                for pair in components if len(pair) == 2
            )
            # mask = [t[0] <= cell_code <= t[1] for t in rng_limits]
            # print(f'{cell_code=}, {mask=}')
            bflag = any(t[0][0] <= cell_code[0] <= t[1][0] and t[0][1] <= cell_code[1] <= t[1][1] for t in rng_limits)
            return bflag
        return True


In [ ]:
exec_time(xlslice_in_fml, is_cell_in_fml, 'C40', '=SUM(A1:D50) + B1 + C1')

xlslice_in_fml:    21.94 µs/call
is_cell_in_fml:    11.92 µs/call


In [3]:
fml = "SUM('sheet1'!A1:D50) + B1 + C1"
dmy = [XlRef(x).xlslice_corners() for x in set(rgn_pattern.findall(fml))]

In [4]:
list(set(itertools.chain(*dmy)))

["'sheet1'!D50", "'sheet1'!A1", 'B1', 'C1']

In [6]:
ws = wb['#sheet2']

In [7]:
tbl = ws.tables[1]
tbl.get_formula()

col,E,F,G,H
row,,,,
12,tabla 2:,,,
13,l1,25,=+G6,=+$G$2*F13
14,l2,10,25,=+SUM(F14:G14)
15,l3,15,38,=+SUM(F15:G15)
16,l4,80,=+F9 + F13,=+SUM(F16:G16)
17,,=+F16+F15+F14,=+G16+G15+G14,=+H16+H15+H14


In [8]:
fmls = tbl.data.loc[~tbl.data.fml.eq('')].fml.to_dict()
fmls = {key: ExcelTable.encoder('decode', fml, df=tbl.data) for key, fml in fmls.items()}

In [13]:
[XlRef(x).xlslice_corners() for fml in fmls.values() for x in rgn_pattern.findall(fml)]

[('$G$2',),
 ('F13',),
 ('F15', 'G15'),
 ('G16',),
 ('G15',),
 ('G14',),
 ('F14', 'G14'),
 ('F16',),
 ('F15',),
 ('F14',),
 ('F9',),
 ('F13',),
 ('H16',),
 ('H15',),
 ('H14',),
 ('F16', 'G16'),
 ('G6',)]

In [15]:
cells_in_fmls = set(itertools.chain(*[XlRef(x).xlslice_corners() for fml in fmls.values() for x in rgn_pattern.findall(fml.replace('$', ''))]))
cells_in_fmls

{'F13',
 'F14',
 'F15',
 'F16',
 'F9',
 'G14',
 'G15',
 'G16',
 'G2',
 'G6',
 'H14',
 'H15',
 'H16'}

In [59]:
def cell_list(fml):
    if not fml:
        return []
    fml = fml.replace('$', '')
    it = rgn_pattern.finditer(fml)
    components = set()
    for m in it:
        sht, xlslice = tbl_pattern.match(m[0]).groups()
        prefix = f"'{sht}'!" if sht else ''
        components.update(prefix + x for x in xlslice.split(':'))
    return components


In [60]:
fmls = ['=SUM(A1:D50) + B1 + C1', "=AV('Sht1'!A1:A10) + B25"]
for fml in fmls:
    print(f'{fml}  {cell_list(fml)}')

=SUM(A1:D50) + B1 + C1  {'A1', 'B1', 'D50', 'C1'}
=AV('Sht1'!A1:A10) + B25  {"'Sht1'!A10", 'B25', "'Sht1'!A1"}


In [68]:
set(itertools.chain(*[cell_list(x) for x in fmls]))

{"'Sht1'!A1", "'Sht1'!A10", 'A1', 'B1', 'B25', 'C1', 'D50'}

In [61]:
def cell_list_new(fml):
    components = [XlRef(x).xlslice_corners() for x in rgn_pattern.findall(fml.replace('$', ''))]
    return set(itertools.chain(*components))

In [62]:
cell_list('=SUM(A1:D50) + B1 + C1')

{'A1', 'B1', 'C1', 'D50'}

In [63]:
exec_time(cell_list, cell_list_new, '=SUM(A1:D50) + B1 + C1')

cell_list:    11.20 µs/call
cell_list_new:    28.55 µs/call


In [70]:
cells = tbl.data.index.tolist()

In [100]:
answer = {}
for cell in cells:
    # dependents = set(fcell for fcell, fml in fmls.items() if is_cell_in_fml(cell, fml))
    dependents = set(fcell for fcell, fml in fmls.items() if xlslice_in_fml(cell, fml))
    if dependents:
        answer[cell] = dependents
answer 

{'G16': {'G17', 'H16'},
 'H14': {'H17'},
 'H15': {'H17'},
 'H16': {'H17'},
 'G14': {'G17', 'H14'},
 'F13': {'G16', 'H13'},
 'G15': {'G17', 'H15'},
 'F15': {'F17', 'H15'},
 'F16': {'F17', 'H16'},
 'F14': {'F17', 'H14'},
 'G6': {'G13'},
 'F9': {'G16'},
 'G2': {'H13'}}

In [93]:
required = dict([(cell, set(ExcelTable.encoder('decode', code, df=tbl.data) for code in dependents)) for cell, dependents in tbl.data.dependents.items() if dependents])
required

{'G16': {'G17', 'H16'},
 'H14': {'H17'},
 'H15': {'H17'},
 'H16': {'H17'},
 'G14': {'G17', 'H14'},
 'F13': {'G16', 'H13'},
 'G15': {'G17', 'H15'},
 'F15': {'F17', 'H15'},
 'F16': {'F17', 'H16'},
 'F14': {'F17', 'H14'},
 'G6': {'G13'},
 'F9': {'G16'},
 'G2': {'H13'}}

In [101]:
answer == required

True

In [6]:
xlslice_in_fml('C40', '=SUM(A1:$D50) + B1 + C1')

True

In [17]:
wb = ExcelWorkbook('MyBook')
sht1 = wb.create_worksheet('Sht1')
tbl1 = sht1.create_table('tbl1', 'A1:P100')
sht2 = wb.create_worksheet('Sht2')
tbl2 = sht2.create_table('tbl2', 'A1:P100')

In [ ]:
import pandas as pd

: 

In [ ]:
fmls = {'A1': "=SUM('Sht2'!A1:A10)"}
fmls_df = pd.Series(fmls).

In [8]:
tbl1.set_formulas({'A1': "=SUM('Sht2'!A1:A10)"})


KeyError: 'A10'

In [14]:
tbl1.data

,fml,dependents,res_order,ftype,value,code
cell,,,,,,
A1,=SUM(tbl2!A1:A2),{A1},1,$,,A1
A10,,{A1},0,#,,A2


In [ ]:
tbl1.set_values({'A10': 50}, recalc=True)

In [11]:
tbl1.data

,fml,dependents,res_order,ftype,value,code
cell,,,,,,
A1,=SUM(tbl2!A1:A2),{A1},1,$,#UNKNOWN!,A1
A10,,{A1},0,#,50,A2
ZZZ8,,{A1},0,$,#UNKNOWN!,'sheet1'!ZZZZ8
